# Notebook n°2 : Analyse exploratoire

Problématique : Dans quelle mesure la structure du réseau TGV est-elle associée à la régularité des liaisons, et peut-on exploiter cette structure pour améliorer la prédiction des retards ?

Ce notebook fait suite au notebook n°1 (qualité des données, Marius).

## 1. Cadrage de l'étude

### 1.1 Type de question

Notre problematique repose principalement sur deux approches:

L'existence d'une relation entre la structure du reseau TGV et la regularite des liaisons et cette approche est exploratoire.

Essayer de determiner si les caracteristiques du reseau peuvent ameliorer la prediction des retards et cette approche est predictive. Notre etude n'est pas causal, on regarde des relations entre les caracteristiques du reseau et les retards mais on cherche pas a savoir si l'un provoque l'autre.

### 1.2 D'où viennent les données ?

Nos données publiées par la SNCF ne couvrent pas la totalité du réseau TGV du territoire français, mais uniquement les liaisons suivies par l'AQST. Certains services sont exclus. De plus, on a des liaisons saisonnières qui ne sont pas présentes toute l'année.

Les retards sont mesurés et communiqués par la SNCF elle-même. Les données sont regroupées par mois et non par jour, elles sont donc déjà agrégées et ne donnent pas de détails sur chaque circulation.


In [1]:
"""Chargement des données SNCF depuis le fichier figé du dépôt."""

from pathlib import Path

import pandas as pd

SEED = 42  


project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

DATA_PATH = project_root / "data" / "brut" / "regularite-mensuelle-tgv-aqst.csv"

data = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig")

print("Dimensions :", data.shape)
data.head()

Dimensions : (12907, 26)


,Date,Service,Gare de départ,Gare d'arrivée,Durée moyenne du trajet,Nombre de circulations prévues,Nombre de trains annulés,Commentaire annulations,Nombre de trains en retard au départ,Retard moyen des trains en retard au départ,...,Nombre trains en retard > 15min,Retard moyen trains en retard > 15 (si liaison concurrencée par vol),Nombre trains en retard > 30min,Nombre trains en retard > 60min,Prct retard pour causes externes,Prct retard pour cause infrastructure,Prct retard pour cause gestion trafic,Prct retard pour cause matériel roulant,Prct retard pour cause gestion en gare et réutilisation de matériel,"Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141,870,5,NaN,289,11.247809,...,110,6.511118,44,8,36.134454,31.092437,10.924370,15.966387,5.042017,0.840336
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56,406,1,NaN,213,8.479969,...,32,5.363539,9,4,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166,226,0,NaN,21,6.239683,...,11,2.938053,6,1,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111
3,2018-01,National,PARIS MONTPARNASSE,NANTES,124,508,3,NaN,71,7.235211,...,39,5.292211,18,8,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94,472,4,NaN,224,6.784673,...,42,4.882372,10,0,15.789474,45.614035,19.298246,15.789474,1.754386,1.754386


In [2]:
# Correspondance entre les noms du fichier CSV et les noms courts
# renvoyés par l'API SNCF, utilisés par les fonctions de Marius (quality.py).
COLUMN_NAMES = {
    "Date": "date",
    "Service": "service",
    "Gare de départ": "gare_depart",
    "Gare d'arrivée": "gare_arrivee",
    "Durée moyenne du trajet": "duree_moyenne",
    "Nombre de circulations prévues": "nb_train_prevu",
    "Nombre de trains annulés": "nb_annulation",
    "Commentaire annulations": "commentaire_annulation",
    "Nombre de trains en retard au départ": "nb_train_depart_retard",
    "Retard moyen des trains en retard au départ": "retard_moyen_depart",
    "Retard moyen de tous les trains au départ": "retard_moyen_tous_trains_depart",
    "Commentaire retards au départ": "commentaire_retards_depart",
    "Nombre de trains en retard à l'arrivée": "nb_train_retard_arrivee",
    "Retard moyen des trains en retard à l'arrivée": "retard_moyen_arrivee",
    "Retard moyen de tous les trains à l'arrivée": "retard_moyen_tous_trains_arrivee",
    "Commentaire retards à l'arrivée": "commentaires_retard_arrivee",
    "Nombre trains en retard > 15min": "nb_train_retard_sup_15",
    "Retard moyen trains en retard > 15 (si liaison concurrencée par vol)": "retard_moyen_trains_retard_sup15",
    "Nombre trains en retard > 30min": "nb_train_retard_sup_30",
    "Nombre trains en retard > 60min": "nb_train_retard_sup_60",
    "Prct retard pour causes externes": "prct_cause_externe",
    "Prct retard pour cause infrastructure": "prct_cause_infra",
    "Prct retard pour cause gestion trafic": "prct_cause_gestion_trafic",
    "Prct retard pour cause matériel roulant": "prct_cause_materiel_roulant",
    "Prct retard pour cause gestion en gare et réutilisation de matériel": "prct_cause_gestion_gare",
    "Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)": "prct_cause_prise_en_charge_voyageurs",
}

# Vérifier que toutes les colonnes du fichier sont bien prévues,
# pour détecter tout de suite un changement de format de la SNCF.
colonnes_inconnues = set(data.columns) - set(COLUMN_NAMES)
assert not colonnes_inconnues, f"Colonnes inattendues : {colonnes_inconnues}"

data = data.rename(columns=COLUMN_NAMES)
data.columns.tolist()

['date',
 'service',
 'gare_depart',
 'gare_arrivee',
 'duree_moyenne',
 'nb_train_prevu',
 'nb_annulation',
 'commentaire_annulation',
 'nb_train_depart_retard',
 'retard_moyen_depart',
 'retard_moyen_tous_trains_depart',
 'commentaire_retards_depart',
 'nb_train_retard_arrivee',
 'retard_moyen_arrivee',
 'retard_moyen_tous_trains_arrivee',
 'commentaires_retard_arrivee',
 'nb_train_retard_sup_15',
 'retard_moyen_trains_retard_sup15',
 'nb_train_retard_sup_30',
 'nb_train_retard_sup_60',
 'prct_cause_externe',
 'prct_cause_infra',
 'prct_cause_gestion_trafic',
 'prct_cause_materiel_roulant',
 'prct_cause_gestion_gare',
 'prct_cause_prise_en_charge_voyageurs']

In [3]:
# Le notebook n°1 a vérifié que la clé (date, service, gare_depart,
# gare_arrivee) est unique : 0 doublon. On vérifie ici pourquoi
# le service doit faire partie de la clé.
doublons_sans_service = data.duplicated(
    subset=["date", "gare_depart", "gare_arrivee"], keep=False
)
print("Lignes en double si on ignore le service :", doublons_sans_service.sum())
print(data.loc[doublons_sans_service, "service"].value_counts())

data["liaison"] = data["gare_depart"] + " → " + data["gare_arrivee"]
gares = pd.concat([data["gare_depart"], data["gare_arrivee"]])

print("\nPériode            :", data["date"].min(), "→", data["date"].max())
print("Nombre de mois     :", data["date"].nunique())
print("Nombre de liaisons :", data["liaison"].nunique())
print("Nombre de gares    :", gares.nunique())

print("\nNombre de lignes par mois :")
data.groupby("date").size().describe()

Lignes en double si on ignore le service : 84
service
International    42
National         42
Name: count, dtype: int64

Période            : 2018-01 → 2026-09
Nombre de mois     : 105
Nombre de liaisons : 130
Nombre de gares    : 59

Nombre de lignes par mois :


count    105.000000
mean     122.923810
std        3.569893
min      118.000000
25%      121.000000
50%      121.000000
75%      123.000000
max      135.000000
dtype: float64

In [4]:
# Quand apparaissent les liaisons publiées en double (National + International) ?
paires = data.loc[doublons_sans_service, ["date", "liaison"]].drop_duplicates()
print("Première date :", paires["date"].min(), "| dernière :", paires["date"].max())
print(paires["liaison"].value_counts())

Première date : 2025-07 | dernière : 2025-09
liaison
PARIS LYON → MACON LOCHE                  3
BELLEGARDE (AIN) → PARIS LYON             3
MACON LOCHE → PARIS LYON                  3
MARSEILLE ST CHARLES → LYON PART DIEU     3
MULHOUSE VILLE → PARIS LYON               3
PARIS EST → STRASBOURG                    3
PARIS LYON → BELLEGARDE (AIN)             3
PARIS LYON → CHAMBERY CHALLES LES EAUX    3
STRASBOURG → PARIS EST                    3
CHAMBERY CHALLES LES EAUX → PARIS LYON    3
DIJON VILLE → PARIS LYON                  3
PARIS LYON → DIJON VILLE                  3
PARIS LYON → MULHOUSE VILLE               3
LYON PART DIEU → MARSEILLE ST CHARLES     3
Name: count, dtype: int64


In [5]:
# Quels mois ont le plus et le moins de lignes ?
lignes_par_mois = data.groupby("date").size()
print(lignes_par_mois.sort_values().head(5))
print(lignes_par_mois.sort_values().tail(5))

date
2021-04    118
2023-10    119
2023-12    119
2023-11    119
2024-06    119
dtype: int64
date
2018-09    130
2018-10    130
2025-07    135
2025-08    135
2025-09    135
dtype: int64



### Premiers observations

Notre jeu de données couvre 105 mois, de janvier 2018 à septembre 2026. Au total, nous avons 130 liaisons et 59 gares.

Le nombre de lignes varie selon les mois. En 2018, on avait 130 observations par mois, contre 121 sur une grande partie des periodes suivantes. Elle a baisse  118 en avril 2021 et a 119 certains mois de 2023 et 2024. Ces variations pourraient etre liees a des evenements comme le Covid. Ceci n'est pas confirme avec nos donnees.

Entre juillet et septembre 2025, 14 liaisons apparaissent à la fois en National et en International, ce qui explique en partie les 135 lignes par mois. Ces observations ne sont pas de vrais doublons puisque le service est différent. Il faut donc prendre en compte le mois, la liaison et le service pour identifier chaque observation.

Comme le nombre d'observations change au cours du temps, nous allons privilégier les taux de retard plutôt que les nombres bruts, afin de faire des comparaisons plus pertinentes.


## Le taux de retard

On cible  pour la suite une seule métrique: la part des trains en retard par liaison et par mois. 
on definit: $$ taux = \frac{\text{nombre de trains en retard}}{\text{nombre de trains ayant circulé}} $$
Au dénominateur, on prend les trains ayant circulé, pas les trains prévus : un train annulé n’a pas roulé, il n’est donc ni à l’heure, ni en retard.

Chaque ligne correspond à une liaison pendant un mois, mais le nombre de trains peut être différent d'une liaison à une autre. Si on fait simplement la moyenne des taux de retard, on donne la même importance à toutes les liaisons, même si certaines ont beaucoup plus de trains que d'autres.

Pour éviter cela, on utilisera un taux pondéré, calculé en divisant le nombre total de trains en retard par le nombre total de trains ayant circulé.

In [6]:

# On calcule le nombre de trains qui ont réellement circulé
data["circules"] = data["nb_train_prevu"] - data["nb_annulation"]

# On garde uniquement les lignes où au moins un train a circulé
data_propre = data[data["circules"] > 0].copy()
print(f"Lignes retirées : {len(data) - len(data_propre)}")


# Fonction pour calculer le taux de retard pondéré
def taux(df, colonne_retards="nb_train_retard_sup_15"):
    return df[colonne_retards].sum() / df["circules"].sum() * 100


# On compare la moyenne simple des taux avec le taux pondéré
taux_naif = (data_propre["nb_train_retard_sup_15"] / data_propre["circules"] * 100).mean()

taux_pondere = taux(data_propre)

print(f"Moyenne simple des taux : {taux_naif:.2f} %")
print(f"Taux pondéré            : {taux_pondere:.2f} %")


Lignes retirées : 73
Moyenne simple des taux : 11.47 %
Taux pondéré            : 10.60 %


## Deux mesures du retard dans le même fichier

jusqu'à présent, nous avons utilisé nb_train_retard_sup_15: le nombre de trains arrivés avec plus de 15 min de retard.

Mais on a la colonne nb_train_retard_arrivee, qui compte elle aussi les trains en retard, mais avec un seuil qui dépend de la durée du trajet. Par exemple pour les trajets de moins de 1h30, un train est en retard a partir de 5 minutes, contre 10 minutes pour les trajets entre 1h30 et 3h, et 15 minutes pour les trajets de plus de 3h. Cela signifie que pour les longs trajets les deux colonnes utilisent le même seuil. 

Les trajets plus courts ont un seuil plus faible avec nb_train_retard_arrivee.

Le choix de la colonne peut donc influencer nos résultats. Nous allons comparer les deux mesures pour voir les différences.

In [7]:
# On découpe les liaisons en 4 classes de durée du trajet.
data_propre["classe_duree"] = pd.cut(
    data_propre["duree_moyenne"],
    bins=[0, 90, 180, 300, 1000],
    labels=["< 1h30", "1h30 - 3h", "3h - 5h", "> 5h"],
)

# Pour chaque classe, on compare les deux mesures de retard.
comparaison = data_propre.groupby("classe_duree", observed=True).apply(lambda g: pd.Series({
    "trains (millions)":     g["circules"].sum() / 1e6,
    "taux officiel (%)":     taux(g, "nb_train_retard_arrivee"),
    "taux > 15 min (%)":     taux(g, "nb_train_retard_sup_15"),
    "rapport > 15 / officiel": g["nb_train_retard_sup_15"].sum() / g["nb_train_retard_arrivee"].sum(),
}))

comparaison.round(2)

,trains (millions),taux officiel (%),taux > 15 min (%),rapport > 15 / officiel
classe_duree,,,,
< 1h30,0.65,15.47,6.43,0.42
1h30 - 3h,1.83,14.18,10.01,0.71
3h - 5h,0.82,14.65,14.15,0.97
> 5h,0.13,17.42,17.43,1.00


### Ce qu'on observe avec les deux mesures

Pour les trajets de plus de 5h on remarque que les deux colonnes donnent les mêmes résultats. C'est cohérent avec la règle SNCF (source : la documentation du dataset sur ressources.data.sncf.com), car pour les longs trajets, le seuil officiel est de 15 minutes.

Tandis que pour les trajets plus courts, le rapport peut descendre à 0,42. Cela s'explique par le fait que le seuil officiel est plus faible pour ces trajets (5 ou 10 minutes).

On remarque aussi que les taux de retard officiels sont assez proches pour toutes les durées, entre 14 et 17 %. Par contre, avec un seuil fixe de 15 minutes, le taux passe d'environ 6 % pour les trajets courts à 17 % pour les trajets très longs.

Ensuite, nous allons utiliser le taux de retard supérieur à 15 minutes comme indicateur principal, afin de comparer les liaisons avec le même seuil. Le taux officiel sera conservé pour vérifier nos résultats.

Il faudra aussi faire attention à la durée des trajets, notamment pour les liaisons internationales ou transversales, afin de ne pas confondre l'effet de la durée avec celui de la structure du réseau.

Il faut aussi noter qu'un même TGV peut apparaître dans plusieurs liaisons. Ainsi, lorsque nous regroupons plusieurs liaisons pour calculer un taux global, certains trains peuvent être comptés plusieurs fois, ce qui peut influencer nos résultats.